# IMDB Hyperparameter Tuning Study

The notebook is for Assignment 1 in 64061.  Each section explains *why* we are doing something before showing the code.

## Step 1: Import Tools
Load the Python libraries to address the following:
- TensorFlow/Keras: build and train neural networks
- NumPy: work with numbers and arrays
- Pandas: organize results into tables
- Matplotlib: create charts

In [ ]:
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, regularizers
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt


## Step 2: Load the IMDB Movie Review Dataset
The dataset contains movie reviews labeled as positive or negative.

Our goal is to predict sentiment using different neural-network designs and compare the results.

In [ ]:
(train_data, train_labels), (test_data, test_labels) = keras.datasets.imdb.load_data(num_words=10000)


## Step 3: Convert Reviews into Numbers
Neural networks cannot read text directly.

Each review is converted into a 10,000-element vector. If a word appears in the review, its position becomes 1. Otherwise it remains 0.

In [ ]:
def vectorize_sequences(sequences, dimension=10000):
    results = np.zeros((len(sequences), dimension))
    for i, sequence in enumerate(sequences):
        results[i, sequence] = 1.0
    return results

x_train = vectorize_sequences(train_data)
x_test = vectorize_sequences(test_data)

y_train = np.asarray(train_labels).astype("float32")
y_test = np.asarray(test_labels).astype("float32")


## Step 4: Create a Validation Set
We reserve 10,000 reviews that the model does not use for learning.

This allows us to measure how well the model generalizes to new information.

In [ ]:
x_val = x_train[:10000]
partial_x_train = x_train[10000:]

y_val = y_train[:10000]
partial_y_train = y_train[10000:]


## Step 5: Build a Reusable Neural Network Function
Instead of creating a new model every time, we create one function that lets us change:
- Number of hidden layers
- Number of units
- Activation functions
- Loss functions
- Regularization methods

In [ ]:
def build_model(hidden_layers=[16,16],
                activation="relu",
                loss_function="binary_crossentropy",
                dropout_rate=None,
                l2_penalty=None):

    model = keras.Sequential()

    for i, units in enumerate(hidden_layers):
        kwargs = {}
        if l2_penalty:
            kwargs["kernel_regularizer"] = regularizers.l2(l2_penalty)

        if i == 0:
            model.add(layers.Dense(units, activation=activation, input_shape=(10000,), **kwargs))
        else:
            model.add(layers.Dense(units, activation=activation, **kwargs))

        if dropout_rate:
            model.add(layers.Dropout(dropout_rate))

    model.add(layers.Dense(1, activation="sigmoid"))

    model.compile(optimizer="rmsprop",
                  loss=loss_function,
                  metrics=["accuracy"])

    return model


## Step 6: Define the Experiments
We now create the list of model configurations to compare.

Each experiment changes one design choice so we can see its impact on performance.

In [ ]:
experiments = {
    "1 Hidden Layer": dict(hidden_layers=[16]),
    "2 Hidden Layers": dict(hidden_layers=[16,16]),
    "3 Hidden Layers": dict(hidden_layers=[16,16,16]),
    "32 Units": dict(hidden_layers=[32,32]),
    "64 Units": dict(hidden_layers=[64,64]),
    "MSE Loss": dict(hidden_layers=[16,16], loss_function="mse"),
    "tanh": dict(hidden_layers=[16,16], activation="tanh"),
    "Dropout + L2": dict(hidden_layers=[32,32], dropout_rate=0.5, l2_penalty=0.001)
}


## Step 7: Train Every Model
For each experiment:
1. Build the model
2. Train it
3. Evaluate it on test data
4. Save the results

In [ ]:
results=[]
histories={}

for name, params in experiments.items():
    print(f"Running: {name}")
    model = build_model(**params)

    history = model.fit(
        partial_x_train,
        partial_y_train,
        epochs=10,
        batch_size=512,
        validation_data=(x_val, y_val),
        verbose=0
    )

    loss, acc = model.evaluate(x_test, y_test, verbose=0)

    results.append({
        "Model": name,
        "Test Loss": round(loss,4),
        "Test Accuracy": round(acc,4)
    })

    histories[name] = history


## Step 8: Create a Results Table
Sort models from most accurate to least accurate.

In [ ]:
results_df = pd.DataFrame(results)
results_df = results_df.sort_values("Test Accuracy", ascending=False)
results_df


## Step 9: Visualize Accuracy
This chart makes it easy to compare performance across all experiments.

In [ ]:
plt.figure(figsize=(10,6))
plt.barh(results_df["Model"], results_df["Test Accuracy"])
plt.xlabel("Test Accuracy")
plt.ylabel("Model")
plt.title("IMDB Hyperparameter Comparison")
plt.tight_layout()
plt.show()


## Step 10: Identify the Best Model
The model with the highest test accuracy is automatically selected.

In [ ]:
best_model_name = results_df.iloc[0]["Model"]
print("Best model:", best_model_name)


## Step 11: Compare Training and Validation Accuracy
This chart helps identify overfitting.

- Training Accuracy = performance on reviews used for learning
- Validation Accuracy = performance on reviews not used for learning

In [ ]:
history = histories[best_model_name]

plt.figure(figsize=(10,6))
plt.plot(history.history["accuracy"], label="Training Accuracy")
plt.plot(history.history["val_accuracy"], label="Validation Accuracy")
plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.title(f"Training vs Validation Accuracy: {best_model_name}")
plt.legend()
plt.show()


## Step 12: Export Results
Save all experiment outcomes to a CSV file for use in the written report.

In [ ]:
results_df.to_csv("imdb_hyperparameter_results.csv", index=False)
print("Results saved.")
